In [1]:
#Import the required packages
#Import package pandas for data analysis
import pandas as pd


In [2]:
def build_dqr_table(df, columns):
    """
    Build and save a Data Quality Report table.
    
    Parameters:
    -----------
    df : pandas.DataFrame
    columns : Index
        Features to include
    csv_filename : str
        Output filename
    
    Returns:
    --------
    pandas.DataFrame
        Complete DQR table
    """
    stats = df[columns].describe().T
    perc_missing = 100 * (df[columns].isnull().sum() / len(df))
    cardinality = df[columns].nunique()
    
    dqr_table = pd.concat([
        stats,
        pd.DataFrame(perc_missing, columns=['%missing']),
        pd.DataFrame(cardinality, columns=['cardinality'])
    ], axis=1)
    
    return dqr_table


print("✓ Helper function loaded successfully")

✓ Helper function loaded successfully


In [3]:
def load_data(filename):
    # Reading from a csv file, into a data frame with error handling
    try:
        # If the csv file has any special symbols, it helps to specify the file encoding.
        df = pd.read_csv(filename, 
                        keep_default_na=True, 
                        delimiter=',', 
                        skipinitialspace=True)
        
        print(f"✓ Successfully loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns")
        
        # Show the first few rows in the data frame
        display(df.head(5))
        
    except FileNotFoundError:
        print("ERROR: The file 'ppr-group-25204989-train.csv' was not found.")
        print("Please ensure the file is in the current working directory.")
    except Exception as e:
        print(f"ERROR: An unexpected error occurred: {e}")

    return df

In [4]:
# Validate that the dataframe has data and expected columns
def validate_data(df):
    print("Dataset Validation Checks:")
    print("="*60)

    # Check 1: Dataset is not empty
    if df.empty:
        print("⚠ WARNING: Dataset is empty!")
    else:
        print(f"✓ Dataset contains {df.shape[0]:,} rows")

    # Check 2: Check for duplicate rows
    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        print(f"⚠ WARNING: Found {duplicate_count} duplicate rows")
    else:
        print("✓ No duplicate rows found")

    # Check 3: Check if all columns are unnamed
    unnamed_cols = [col for col in df.columns if 'Unnamed' in str(col)]
    if unnamed_cols:
        print(f"⚠ WARNING: Found {len(unnamed_cols)} unnamed columns: {unnamed_cols}")
    else:
        print("✓ All columns have names")

    # Check 4: Basic column overview
    print(f"\n✓ Dataset has {df.shape[1]} columns:")
    print(f"  - Column names: {list(df.columns)}")

## 1. ALF01.csv

In [5]:
df=load_data("EHQ03.csv")

✓ Successfully loaded dataset with 130536 rows and 10 columns


,STATISTIC,Statistic Label,TLIST(Q1),Quarter,C02665V03225,Economic Sector NACE Rev 2,C02397V02888,Type of Employee,UNIT,VALUE
0,EHQ03C01,Employment,20081,2008Q1,-,All NACE economic sectors,-,All employees,Number,1767200.0
1,EHQ03C01,Employment,20081,2008Q1,-,All NACE economic sectors,1,"Managers, professionals and associated profess...",Number,549400.0
2,EHQ03C01,Employment,20081,2008Q1,-,All NACE economic sectors,2,"Clerical, sales and service employees",Number,685300.0
3,EHQ03C01,Employment,20081,2008Q1,-,All NACE economic sectors,3,"Production, transport, craft and other manual ...",Number,532600.0
4,EHQ03C01,Employment,20081,2008Q1,B,Mining and quarrying (B),-,All employees,Number,7100.0


In [6]:
validate_data(df)

Dataset Validation Checks:
✓ Dataset contains 130,536 rows
✓ No duplicate rows found
✓ All columns have names

✓ Dataset has 10 columns:
  - Column names: ['STATISTIC', 'Statistic Label', 'TLIST(Q1)', 'Quarter', 'C02665V03225', 'Economic Sector NACE Rev 2', 'C02397V02888', 'Type of Employee', 'UNIT', 'VALUE']


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130536 entries, 0 to 130535
Data columns (total 10 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   STATISTIC                   130536 non-null  object 
 1   Statistic Label             130536 non-null  object 
 2   TLIST(Q1)                   130536 non-null  int64  
 3   Quarter                     130536 non-null  object 
 4   C02665V03225                130536 non-null  object 
 5   Economic Sector NACE Rev 2  130536 non-null  object 
 6   C02397V02888                130536 non-null  object 
 7   Type of Employee            130536 non-null  object 
 8   UNIT                        130536 non-null  object 
 9   VALUE                       20028 non-null   float64
dtypes: float64(1), int64(1), object(8)
memory usage: 10.0+ MB


In [8]:


# Convert all object columns to categorical
for col in df.select_dtypes('object').columns:
    df[col] = df[col].astype('category')

df['TLIST(Q1)']=df['TLIST(Q1)'].astype('category')


# Update column groups after type corrections
numeric_columns = df.select_dtypes(['int64', 'float64']).columns
category_columns = df.select_dtypes('category').columns

print("✓ Data types corrected:")
print(f"  - Numeric features: {len(numeric_columns)}")
print(f"  - Categorical features: {len(category_columns)}")

✓ Data types corrected:
  - Numeric features: 1
  - Categorical features: 9


In [9]:
df_numeric_dqr = build_dqr_table(df, numeric_columns)
display(df_numeric_dqr)

,count,mean,std,min,25%,50%,75%,max,%missing,cardinality
VALUE,20028.0,16026.49036,118669.549549,-8.27,10.1675,29.425,690.3675,2368400.0,84.657106,9656


In [10]:
df_categorical_dqr = build_dqr_table(df, category_columns)
display(df_categorical_dqr)

,count,unique,top,freq,%missing,cardinality
STATISTIC,130536,21,EHQ03C01,6216,0.0,21
Statistic Label,130536,21,Average Hourly Benefit in Kind,6216,0.0,21
TLIST(Q1),130536,74,20081,1764,0.0,74
Quarter,130536,74,2008Q1,1764,0.0,74
C02665V03225,130536,21,-,6216,0.0,21
Economic Sector NACE Rev 2,130536,21,Accommodation and food service activities (I),6216,0.0,21
C02397V02888,130536,4,-,32634,0.0,4
Type of Employee,130536,4,All employees,32634,0.0,4
UNIT,130536,3,Euro,111888,0.0,3


In [11]:
# Count each value in every categorical column, including missing values.
for col in category_columns:
    counts = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="count")
    )

    print(f"\n{col}")
    display(counts)


STATISTIC


,value,count
0,EHQ03C01,6216
1,EHQ03S10,6216
2,EHQ03S8,6216
3,EHQ03S7,6216
4,EHQ03S6,6216
5,EHQ03S5,6216
6,EHQ03S4,6216
7,EHQ03S3,6216
8,EHQ03S2,6216
9,EHQ03S11,6216



Statistic Label


,value,count
0,Average Hourly Benefit in Kind,6216
1,Average Hourly Total Labour Costs (Seasonally ...,6216
2,Average weekly earnings excluding irregular ea...,6216
3,Average weekly earnings excluding irregular ea...,6216
4,Average weekly earnings excluding irregular an...,6216
5,Average weekly earnings excluding irregular an...,6216
6,Average Weekly Paid Hours (Seasonally Adjusted),6216
7,Average Weekly Paid Hours,6216
8,Average Weekly Earnings (Seasonally Adjusted),6216
9,Average Weekly Earnings,6216



TLIST(Q1)


...

,value,count
0,-,32634
1,1,32634
2,2,32634
3,3,32634



Type of Employee


,value,count
0,All employees,32634
1,"Clerical, sales and service employees",32634
2,"Managers, professionals and associated profess...",32634
3,"Production, transport, craft and other manual ...",32634



UNIT


,value,count
0,Euro,111888
1,Hours,12432
2,Number,6216


In [12]:
for col in category_columns:
    missing_stats = (
        df.groupby(col, observed=True, dropna=False)["VALUE"]
        .agg(
            total_rows="size",
            valid_rows="count",
        )
    )

    missing_stats["missing_rows"] = (
        missing_stats["total_rows"] - missing_stats["valid_rows"]
    )
    missing_stats["missing_pct"] = (
        missing_stats["missing_rows"]
        .div(missing_stats["total_rows"])
        .mul(100)
        .round(2)
    )

    print(f"\n{col}")
    display(missing_stats.sort_values("missing_pct", ascending=False))


STATISTIC


,total_rows,valid_rows,missing_rows,missing_pct
STATISTIC,,,,
EHQ03S9,6216,0,6216,100.00
EHQ03S10,6216,0,6216,100.00
EHQ03S8,6216,0,6216,100.00
EHQ03S7,6216,0,6216,100.00
EHQ03S6,6216,0,6216,100.00
EHQ03S5,6216,0,6216,100.00
EHQ03S4,6216,0,6216,100.00
EHQ03S3,6216,0,6216,100.00
EHQ03S2,6216,0,6216,100.00



Statistic Label


,total_rows,valid_rows,missing_rows,missing_pct
Statistic Label,,,,
Average Weekly Earnings (Seasonally Adjusted),6216,0,6216,100.00
Average weekly earnings excluding irregular earnings (Seasonally adjusted),6216,0,6216,100.00
Average Hourly Earnings (Seasonally Adjusted),6216,0,6216,100.00
Average Hourly Earnings excluding Irregular Earnings (Seasonally Adjusted),6216,0,6216,100.00
Average weekly earnings excluding irregular and overtime earnings (Seasonally adjusted),6216,0,6216,100.00
Average Hourly Irregular Earnings (Seasonally Adjusted),6216,0,6216,100.00
Average Hourly Other Labour Costs (Seasonally Adjusted),6216,0,6216,100.00
Average Hourly Benefit in Kind (Seasonally Adjusted),6216,0,6216,100.00
Average Hourly Total Labour Costs (Seasonally Adjusted),6216,0,6216,100.00



TLIST(Q1)


...

,total_rows,valid_rows,missing_rows,missing_pct
C02397V02888,,,,
1,32634,1141,31493,96.50
2,32634,1141,31493,96.50
3,32634,1156,31478,96.46
-,32634,16590,16044,49.16



Type of Employee


,total_rows,valid_rows,missing_rows,missing_pct
Type of Employee,,,,
"Clerical, sales and service employees",32634,1141,31493,96.50
"Managers, professionals and associated professionals",32634,1141,31493,96.50
"Production, transport, craft and other manual workers",32634,1156,31478,96.46
All employees,32634,16590,16044,49.16



UNIT


,total_rows,valid_rows,missing_rows,missing_pct
UNIT,,,,
Euro,111888,16212,95676,85.51
Hours,12432,2009,10423,83.84
Number,6216,1807,4409,70.93
